# Logic-LM: tiny neuro-symbolic reasoning demo

Logic-LM separates logical reasoning into **problem formulation**, **symbolic inference**, and **result interpretation**, with an optional **self-refinement** loop driven by solver errors.

This notebook implements those interfaces with a hand-authored symbolic program and a deterministic forward-chaining solver. It uses no LLM, API key, or external package.

Paper: https://aclanthology.org/2023.findings-emnlp.248/  
Official code: https://github.com/teacherpeterpan/Logic-LLM

## 1. Problem formulation

Natural-language problem:

> Metals conduct electricity. If something is made of iron, then it is metal. Nails are made of iron. Is the statement ‘Nails cannot conduct electricity’ true, false, or unknown?

In the paper, an LLM performs this translation. Here the symbolic formulation is written explicitly so the solver behavior remains easy to inspect.

In [ ]:
program = {
    "predicates": ["made_of_iron", "metal", "conducts_electricity"],
    "facts": [
        {"predicate": "made_of_iron", "args": ["nail"], "truth": True}
    ],
    "rules": [
        {
            "premises": [{"predicate": "made_of_iron", "args": ["?x"], "truth": True}],
            "conclusion": {"predicate": "metal", "args": ["?x"], "truth": True},
        },
        {
            "premises": [{"predicate": "metal", "args": ["?x"], "truth": True}],
            "conclusion": {"predicate": "conducts_electricity", "args": ["?x"], "truth": True},
        },
    ],
    # The natural-language query says that nails CANNOT conduct electricity.
    "query": {"predicate": "conducts_electricity", "args": ["nail"], "truth": False},
}
program

In [ ]:
from copy import deepcopy
from difflib import get_close_matches

def atoms(program):
    yield from program["facts"]
    yield program["query"]
    for rule in program["rules"]:
        yield from rule["premises"]
        yield rule["conclusion"]

def validate(program):
    declared = set(program["predicates"])
    used = {atom["predicate"] for atom in atoms(program)}
    unknown = sorted(used - declared)
    if unknown:
        raise ValueError(f"Unknown predicates: {unknown}; declared predicates: {sorted(declared)}")
    for atom in atoms(program):
        if not isinstance(atom.get("truth"), bool):
            raise ValueError(f"truth must be Boolean in atom: {atom}")

def fact_tuple(atom, env=None):
    env = env or {}
    args = tuple(env.get(arg, arg) for arg in atom["args"])
    return atom["predicate"], args, atom["truth"]

def unify(pattern_args, fact_args, env):
    new_env = dict(env)
    for pattern, value in zip(pattern_args, fact_args):
        if pattern.startswith("?"):
            if pattern in new_env and new_env[pattern] != value:
                return None
            new_env[pattern] = value
        elif pattern != value:
            return None
    return new_env

def satisfy(premises, known):
    environments = [{}]
    for premise in premises:
        next_environments = []
        for env in environments:
            for predicate, args, truth in known:
                if predicate == premise["predicate"] and truth == premise["truth"]:
                    matched = unify(premise["args"], args, env)
                    if matched is not None:
                        next_environments.append(matched)
        environments = next_environments
    return environments

def solve(program):
    validate(program)
    known = {fact_tuple(fact) for fact in program["facts"]}
    proof = []
    changed = True
    while changed:
        changed = False
        for rule_id, rule in enumerate(program["rules"]):
            for env in satisfy(rule["premises"], known):
                conclusion = fact_tuple(rule["conclusion"], env)
                if conclusion not in known:
                    known.add(conclusion)
                    proof.append({"rule_id": rule_id, "bindings": env, "derived": conclusion})
                    changed = True

    query = fact_tuple(program["query"])
    opposite = (query[0], query[1], not query[2])
    label = "true" if query in known else "false" if opposite in known else "unknown"
    return {"label": label, "known_facts": sorted(known), "proof": proof}

print("Symbolic solver ready.")

In [ ]:
# 2. Symbolic reasoning and 3. result interpretation.
result = solve(program)
print("Derived facts:")
for fact in result["known_facts"]:
    print(" ", fact)
print("\nProof trace:")
for step in result["proof"]:
    print(" ", step)
print("\nQuery label:", result["label"])
print("Interpretation: The statement 'Nails cannot conduct electricity' is false.")
assert result["label"] == "false"

## 4. Solver-feedback self-refinement

The next program imitates an imperfect formulation: the conclusion contains the misspelled predicate `conducts_electricty`. Validation returns a concrete error, which a refinement step uses to align the spelling with the declared vocabulary before retrying.

In [ ]:
broken_program = deepcopy(program)
broken_program["rules"][1]["conclusion"]["predicate"] = "conducts_electricty"

try:
    solve(broken_program)
except ValueError as error:
    solver_feedback = str(error)
    print("Solver feedback:", solver_feedback)

In [ ]:
def refine_unknown_predicates(program):
    repaired = deepcopy(program)
    declared = repaired["predicates"]
    repairs = {}
    for atom in atoms(repaired):
        predicate = atom["predicate"]
        if predicate not in declared:
            candidates = get_close_matches(predicate, declared, n=1, cutoff=0.6)
            if not candidates:
                raise ValueError(f"No safe repair found for {predicate!r}")
            repairs[predicate] = candidates[0]
            atom["predicate"] = candidates[0]
    return repaired, repairs

refined_program, repairs = refine_unknown_predicates(broken_program)
refined_result = solve(refined_program)
print("Repairs:", repairs)
print("Label after refinement:", refined_result["label"])
assert repairs == {"conducts_electricty": "conducts_electricity"}
assert refined_result["label"] == "false"

## Scope

The complete Logic-LM system uses an LLM to formalize problems and routes different datasets to logic-programming, first-order-logic, constraint, or SAT/SMT solvers. This notebook isolates the key contract: a probabilistic formulator proposes a symbolic representation, while a deterministic solver owns inference and supplies actionable errors when the representation is invalid.